# 2024 Bahrain Grand Prix — F1 Pitwall Analysis

This notebook walks through the full F1 Pitwall analytics pipeline applied to the 2024 Bahrain Grand Prix — Round 1 of the season.

**What we cover:**
1. Loading and parsing session data via FastF1
2. Driver performance ratings across 5 dimensions
3. Stint-by-stint strategy breakdown
4. Lap time and degradation analysis
5. Driver DNA clustering and similarity map

---

## 0. Setup

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import rcParams
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# F1 Pitwall modules
from f1pitwall.data_pipeline import (
    load_session, get_session_info,
    parse_all_drivers, load_parsed_session, save_parsed_session,
)
from f1pitwall.scoring_engine import compute_race_ratings
from f1pitwall.scoring_engine.qualifying_perf import score_qualifying_perf
from f1pitwall.strategy_simulator import build_strategy_reports
from f1pitwall.driver_dna import (
    build_dna_vectors, cluster_drivers,
    build_similarity_ranking, build_dna_profiles,
    find_optimal_clusters, FEATURE_NAMES,
)

# Styling
rcParams['figure.facecolor'] = '#0f0f0f'
rcParams['axes.facecolor']   = '#1a1a1a'
rcParams['axes.edgecolor']   = '#444'
rcParams['text.color']       = '#ffffff'
rcParams['axes.labelcolor']  = '#ffffff'
rcParams['xtick.color']      = '#aaaaaa'
rcParams['ytick.color']      = '#aaaaaa'
rcParams['grid.color']       = '#333333'
rcParams['grid.linestyle']   = '--'
rcParams['grid.alpha']       = 0.5
rcParams['font.family']      = 'monospace'

SEASON, ROUND = 2024, 1
print(f'Setup complete. Analysing: {SEASON} Round {ROUND}')

Setup complete. Analysing: 2024 Round 1


---
## 1. Load Session Data

FastF1 pulls timing, telemetry, and weather data from the F1 data feed.
After the first load everything is cached locally — subsequent runs are instant.

In [ ]:
# Load race session
session_race = load_session(SEASON, ROUND, 'R', load_telemetry=True)
race_info    = get_session_info(session_race)

# Load qualifying session
session_quali = load_session(SEASON, ROUND, 'Q', load_telemetry=False)

print(f'Event:   {session_race.event["EventName"]}')
print(f'Circuit: {race_info.circuit}, {race_info.country}')
print(f'Date:    {race_info.date.strftime("%d %B %Y")}')
print(f'Session: {race_info.session_type}')

In [ ]:
# Parse all drivers — uses local cache if available
race_data = load_parsed_session(SEASON, ROUND, 'R')
if not race_data:
    print('Parsing race data...')
    race_data = parse_all_drivers(session_race)
    save_parsed_session(SEASON, ROUND, 'R', race_data)

quali_data = load_parsed_session(SEASON, ROUND, 'Q')
if not quali_data:
    print('Parsing quali data...')
    quali_data = parse_all_drivers(session_quali)
    save_parsed_session(SEASON, ROUND, 'Q', quali_data)

total_laps = max(l.lap_number for d in race_data.values() for l in d.laps)

print(f'\nParsed {len(race_data)} drivers')
print(f'Total race laps: {total_laps}')
print(f'\nDrivers: {", ".join(sorted(race_data.keys()))}')

In [ ]:
# Quick data summary
summary_rows = []
for code, driver in sorted(race_data.items()):
    summary_rows.append({
        'Code':         code,
        'Driver':       driver.full_name,
        'Team':         driver.team,
        'Grid':         driver.grid_position,
        'Finish':       driver.finish_position,
        'Valid Laps':   len(driver.valid_laps),
        'Stints':       len(driver.stints),
        'Fastest (s)':  round(driver.fastest_lap_s, 3) if driver.fastest_lap_s else None,
    })

df_summary = pd.DataFrame(summary_rows).sort_values('Finish')
df_summary

---
## 2. Driver Performance Ratings

Each driver is scored across 5 dimensions:
- **Race Craft** — positions gained, overtake index
- **Pace Efficiency** — lap time delta vs teammate and session best
- **Tyre Management** — degradation slope, late-stint stability
- **Consistency** — weighted lap time standard deviation
- **Qualifying** — best lap delta to pole and teammate

All scores are normalised 0–100 across the field.

In [ ]:
# Compute ratings
quali_scores = None
if quali_data:
    raw_quali    = score_qualifying_perf(quali_data)
    quali_scores = {c: raw_quali[c] for c in race_data if c in raw_quali}

ratings = compute_race_ratings(race_data, session_info=race_info, quali_scores=quali_scores)

# Build leaderboard DataFrame
rows = []
for rank, (code, r) in enumerate(ratings.items(), 1):
    p = r.pillars
    rows.append({
        'Rank':        rank,
        'Code':        code,
        'Driver':      r.full_name,
        'Team':        r.team,
        'Race Craft':  round(p['Race Craft'].score, 1)      if 'Race Craft'      in p else None,
        'Pace Eff.':   round(p['Pace Efficiency'].score, 1) if 'Pace Efficiency' in p else None,
        'Tyre Mgmt':   round(p['Tyre Management'].score, 1) if 'Tyre Management' in p else None,
        'Consistency': round(p['Consistency'].score, 1)     if 'Consistency'     in p else None,
        'Qualifying':  round(p['Qualifying'].score, 1)      if 'Qualifying'      in p else None,
        'TOTAL':       r.composite_score,
    })

df_ratings = pd.DataFrame(rows)
df_ratings.style.background_gradient(
    subset=['Race Craft', 'Pace Eff.', 'Tyre Mgmt', 'Consistency', 'Qualifying', 'TOTAL'],
    cmap='RdYlGn', vmin=0, vmax=100
).format(precision=1)

In [ ]:
# Horizontal bar chart — composite scores
df_plot = df_ratings.sort_values('TOTAL')

fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.RdYlGn(
    [(s - df_plot['TOTAL'].min()) / (df_plot['TOTAL'].max() - df_plot['TOTAL'].min())
     for s in df_plot['TOTAL']]
)

bars = ax.barh(df_plot['Code'], df_plot['TOTAL'], color=colors, edgecolor='#333', linewidth=0.5)

for bar, score in zip(bars, df_plot['TOTAL']):
    ax.text(
        bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
        f'{score:.1f}', va='center', ha='left', fontsize=9, color='#ffffff'
    )

ax.set_xlabel('Composite Score', fontsize=11)
ax.set_title('2024 Bahrain GP — Driver Ratings', fontsize=14, pad=15)
ax.set_xlim(0, 110)
ax.grid(axis='x')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('notebooks/bahrain_ratings.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

In [ ]:
# Radar chart — top 3 drivers
PILLARS = ['Race Craft', 'Pace Eff.', 'Tyre Mgmt', 'Consistency', 'Qualifying']
TEAM_COLORS = {
    'Red Bull Racing': '#3671C6', 'Ferrari': '#E8002D',
    'Mercedes': '#27F4D2', 'McLaren': '#FF8000',
    'Aston Martin': '#229971', 'Alpine': '#FF87BC',
    'Williams': '#64C4FF', 'RB': '#6692FF',
    'Kick Sauber': '#52E252', 'Haas F1 Team': '#B6BABD',
}

top3_codes = list(ratings.keys())[:3]

fig = go.Figure()
for code in top3_codes:
    r = ratings[code]
    vals = [
        r.pillars.get('Race Craft',      type('', (), {'score': 0})()).score,
        r.pillars.get('Pace Efficiency', type('', (), {'score': 0})()).score,
        r.pillars.get('Tyre Management', type('', (), {'score': 0})()).score,
        r.pillars.get('Consistency',     type('', (), {'score': 0})()).score,
        r.pillars.get('Qualifying',      type('', (), {'score': 0})()).score,
    ]
    vals_closed  = vals + [vals[0]]
    labels_closed = PILLARS + [PILLARS[0]]
    color = TEAM_COLORS.get(r.team, '#888888')

    fig.add_trace(go.Scatterpolar(
        r=vals_closed, theta=labels_closed,
        fill='toself', fillcolor=color, opacity=0.2,
        line=dict(color=color, width=2),
        name=f'{r.full_name} — {r.composite_score:.1f}'
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title='Top 3 Drivers — Performance Radar',
    height=500,
    paper_bgcolor='#0f0f0f',
    font=dict(color='#ffffff'),
)
fig.show()

---
## 3. Strategy Analysis

Reconstructing the race strategy for every driver — pit stop timing,
tyre compounds, optimal windows, and safety car luck.

In [ ]:
# Build strategy reports
reports = build_strategy_reports(race_data, session_race, race_info)
print(f'Strategy reports built for {len(reports)} drivers')

In [ ]:
# Tyre stint visualisation — all drivers
COMPOUND_COLORS = {
    'SOFT': '#FF3333', 'MEDIUM': '#FFD700', 'HARD': '#CCCCCC',
    'INTERMEDIATE': '#39B54A', 'WET': '#0067FF', 'UNKNOWN': '#888888',
}

sorted_drivers = sorted(
    race_data.items(),
    key=lambda x: (x[1].finish_position or 99)
)

fig, ax = plt.subplots(figsize=(14, 9))

yticks, ylabels = [], []

for idx, (code, driver) in enumerate(sorted_drivers):
    y = len(sorted_drivers) - idx
    yticks.append(y)
    ylabels.append(f"P{driver.finish_position or '?'} {code}")

    for stint in driver.stints:
        if not stint.laps:
            continue
        start = stint.laps[0].lap_number
        end   = stint.laps[-1].lap_number
        color = COMPOUND_COLORS.get(stint.compound.value, '#888')

        ax.barh(
            y, end - start + 1, left=start - 1,
            color=color, edgecolor='#111', linewidth=0.5, height=0.7
        )

    # Pit stop markers
    report = reports.get(code)
    if report and report.pit_analysis:
        for ev in report.pit_analysis.pit_events:
            verdict_color = {'optimal': '#00CC66', 'early': '#FFD700', 'late': '#FF4444'}.get(
                ev.timing_verdict, '#fff'
            )
            ax.plot(ev.pit_lap, y, 'v', color=verdict_color, markersize=8, zorder=5)

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels, fontsize=9)
ax.set_xlabel('Lap', fontsize=11)
ax.set_title('2024 Bahrain GP — Tyre Strategy', fontsize=14, pad=15)
ax.set_xlim(0, total_laps + 1)
ax.grid(axis='x')
ax.spines[['top', 'right']].set_visible(False)

# Legend
compound_patches = [
    mpatches.Patch(color=c, label=k)
    for k, c in COMPOUND_COLORS.items() if k != 'UNKNOWN'
]
verdict_patches = [
    mpatches.Patch(color='#00CC66', label='Pit: Optimal'),
    mpatches.Patch(color='#FFD700', label='Pit: Early'),
    mpatches.Patch(color='#FF4444', label='Pit: Late'),
]
ax.legend(
    handles=compound_patches + verdict_patches,
    loc='lower right', fontsize=8, ncol=4,
    facecolor='#1a1a1a', edgecolor='#444'
)

plt.tight_layout()
plt.savefig('notebooks/bahrain_strategy.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

In [ ]:
# Pit stop summary table
pit_rows = []
for code, driver in sorted_drivers:
    report = reports.get(code)
    if not report or not report.pit_analysis:
        continue
    pa = report.pit_analysis
    pit_rows.append({
        'Code':       code,
        'Driver':     driver.full_name,
        'Finish':     driver.finish_position,
        'Stops':      len(pa.pit_events),
        'Pit Loss':   f'{pa.total_pit_loss_s:.1f}s',
        'Strategy':   pa.strategy_summary,
    })

pd.DataFrame(pit_rows)

---
## 4. Lap Time & Degradation Analysis

Visualising lap time evolution across the race and tyre degradation
slopes per stint for selected drivers.

In [ ]:
# Lap time chart — top 5 finishers
top5 = [code for code, _ in sorted_drivers[:5]]

fig = go.Figure()

for code in top5:
    driver = race_data[code]
    valid  = driver.valid_laps
    color  = TEAM_COLORS.get(driver.team, '#888')

    fig.add_trace(go.Scatter(
        x=[l.lap_number for l in valid],
        y=[l.lap_time_s for l in valid],
        mode='lines+markers',
        name=f'{code} — {driver.full_name}',
        line=dict(color=color, width=1.5),
        marker=dict(
            color=[COMPOUND_COLORS.get(l.compound.value, '#888') for l in valid],
            size=5,
        ),
        hovertemplate=(
            f'{code}<br>'
            'Lap %{x}<br>'
            'Time: %{y:.3f}s'
            '<extra></extra>'
        )
    ))

fig.update_layout(
    title='2024 Bahrain GP — Lap Times (Top 5 Finishers)',
    xaxis_title='Lap',
    yaxis_title='Lap Time (s)',
    yaxis=dict(autorange='reversed'),
    height=480,
    hovermode='x unified',
    paper_bgcolor='#0f0f0f',
    plot_bgcolor='#1a1a1a',
    font=dict(color='#ffffff'),
    xaxis=dict(gridcolor='#333'),
    yaxis_gridcolor='#333',
)
fig.show()

In [ ]:
# Degradation slope comparison — all drivers, all stints
deg_rows = []
for code, driver in race_data.items():
    for stint in driver.stints:
        if len(stint.laps) < 4:
            continue
        deg_rows.append({
            'Code':     code,
            'Driver':   driver.full_name,
            'Team':     driver.team,
            'Stint':    stint.stint_number,
            'Compound': stint.compound.value,
            'Laps':     len(stint.laps),
            'Avg Time': round(stint.avg_lap_time_s, 3),
            'Deg Slope (s/lap)': round(stint.degradation_slope, 5),
        })

df_deg = pd.DataFrame(deg_rows).sort_values('Deg Slope (s/lap)')
df_deg.style.background_gradient(
    subset=['Deg Slope (s/lap)'],
    cmap='RdYlGn_r',
).format({'Deg Slope (s/lap)': '{:.5f}', 'Avg Time': '{:.3f}'})

In [ ]:
# Verstappen vs Hamilton — direct stint comparison
COMPARE = ['VER', 'HAM']

fig, axes = plt.subplots(1, len(COMPARE), figsize=(14, 5), sharey=True)

for ax, code in zip(axes, COMPARE):
    driver = race_data[code]
    color  = TEAM_COLORS.get(driver.team, '#888')

    for stint in driver.stints:
        laps  = [l for l in stint.laps if l.lap_time_s]
        if len(laps) < 3:
            continue
        ages  = [l.tyre_life for l in laps]
        times = [l.lap_time_s for l in laps]
        c     = COMPOUND_COLORS.get(stint.compound.value, '#888')

        ax.scatter(ages, times, color=c, s=25, zorder=3,
                   label=f'Stint {stint.stint_number} {stint.compound.value}')

        # Trend line
        z = np.polyfit(ages, times, 1)
        p = np.poly1d(z)
        xs = np.linspace(min(ages), max(ages), 50)
        ax.plot(xs, p(xs), color=c, alpha=0.6, linewidth=1.5, linestyle='--')

        # Slope annotation
        ax.annotate(
            f'+{stint.degradation_slope:.4f}s/lap',
            xy=(max(ages), p(max(ages))),
            xytext=(5, 0), textcoords='offset points',
            fontsize=8, color=c
        )

    ax.set_title(f'{driver.full_name}', fontsize=11)
    ax.set_xlabel('Tyre Life (laps)', fontsize=9)
    if ax == axes[0]:
        ax.set_ylabel('Lap Time (s)', fontsize=9)
    ax.legend(fontsize=8, facecolor='#1a1a1a', edgecolor='#444')
    ax.grid(True)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Tyre Degradation — VER vs HAM', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('notebooks/bahrain_degradation.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

---
## 5. Driver DNA — Clustering & Similarity

Building identity vectors for each driver across 8 dimensions,
then clustering by driving style using KMeans and visualising
in 2D using PCA.

In [ ]:
# Build DNA vectors
dna_vectors = build_dna_vectors(race_data, quali_data)

# Find optimal cluster count
optimal_k, inertias = find_optimal_clusters(dna_vectors, max_k=7)
print(f'Optimal cluster count: {optimal_k}')

# Plot elbow curve
fig, ax = plt.subplots(figsize=(7, 4))
k_values = list(range(2, 2 + len(inertias)))
ax.plot(k_values, inertias, 'o-', color='#4ECDC4', linewidth=2, markersize=7)
ax.axvline(optimal_k, color='#FF6B6B', linestyle='--', alpha=0.7, label=f'Optimal k={optimal_k}')
ax.set_xlabel('Number of Clusters (k)', fontsize=10)
ax.set_ylabel('Inertia', fontsize=10)
ax.set_title('KMeans Elbow Curve — Driver DNA', fontsize=12)
ax.legend(facecolor='#1a1a1a', edgecolor='#444')
ax.grid(True)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Run clustering
clustering = cluster_drivers(dna_vectors, n_clusters=optimal_k)

var1 = clustering.pca_variance_explained[0] * 100
var2 = clustering.pca_variance_explained[1] * 100
print(f'PCA variance explained: PC1={var1:.1f}%  PC2={var2:.1f}%')
print(f'Total: {var1+var2:.1f}%')

print('\nFeature importance for PC1:')
for feat, imp in sorted(clustering.feature_importance.items(), key=lambda x: -x[1]):
    bar = '█' * int(imp * 30)
    print(f'  {feat:<22} {imp:.3f}  {bar}')

In [ ]:
# Driver similarity map — 2D PCA scatter
CLUSTER_COLORS = ['#FF6B6B', '#4ECDC4', '#FFE66D', '#A8E6CF', '#FF8B94', '#B8B8FF']

fig = go.Figure()

from collections import defaultdict
cluster_groups = defaultdict(list)
for code, result in clustering.clusters.items():
    cluster_groups[result.cluster_label].append((code, result))

for i, (label, members) in enumerate(cluster_groups.items()):
    color = CLUSTER_COLORS[i % len(CLUSTER_COLORS)]
    team_colors_list = [TEAM_COLORS.get(r.team, '#888') for _, r in members]

    fig.add_trace(go.Scatter(
        x=[r.pca_x for _, r in members],
        y=[r.pca_y for _, r in members],
        mode='markers+text',
        name=label,
        text=[code for code, _ in members],
        textposition='top center',
        textfont=dict(size=11, color='#ffffff'),
        marker=dict(
            color=team_colors_list,
            size=18,
            line=dict(width=2, color=color),
            opacity=0.9,
        ),
        hovertemplate=(
            '<b>%{text}</b><br>'
            f'Style: {label}<br>'
            'PC1: %{x:.2f}<br>PC2: %{y:.2f}'
            '<extra></extra>'
        ),
    ))

fig.update_layout(
    title=f'Driver Similarity Map — 2024 Bahrain GP<br>'
          f'<sup>PC1={var1:.1f}% variance  PC2={var2:.1f}% variance</sup>',
    xaxis_title=f'PC1 ({var1:.1f}% variance)',
    yaxis_title=f'PC2 ({var2:.1f}% variance)',
    height=580,
    paper_bgcolor='#0f0f0f',
    plot_bgcolor='#1a1a1a',
    font=dict(color='#ffffff'),
    xaxis=dict(gridcolor='#333', zeroline=False),
    yaxis=dict(gridcolor='#333', zeroline=False),
    legend=dict(bgcolor='rgba(0,0,0,0)', title='Driving Style'),
)
fig.show()

In [ ]:
# DNA feature heatmap
sorted_codes = sorted(
    dna_vectors.keys(),
    key=lambda c: (
        clustering.clusters[c].cluster_id if c in clustering.clusters else 99,
        c
    )
)

heatmap_rows = []
for code in sorted_codes:
    dna = dna_vectors[code]
    cluster = clustering.clusters.get(code)
    row = {'Driver': f'{code} [{cluster.cluster_label if cluster else "?"}]'}
    for feat in FEATURE_NAMES:
        row[feat.replace('_', ' ').title()] = dna.features.get(feat, 0.0)
    heatmap_rows.append(row)

df_heatmap = pd.DataFrame(heatmap_rows).set_index('Driver')

fig = px.imshow(
    df_heatmap,
    color_continuous_scale='RdYlGn',
    zmin=0, zmax=1,
    aspect='auto',
    text_auto='.2f',
    title='Driver DNA Feature Heatmap — 2024 Bahrain GP',
)
fig.update_layout(
    height=600,
    paper_bgcolor='#0f0f0f',
    font=dict(color='#ffffff', size=10),
    xaxis=dict(side='top'),
    margin=dict(l=160, r=20, t=100, b=20),
)
fig.update_traces(textfont=dict(size=9))
fig.show()

In [ ]:
# Similarity rankings — who drives most like Verstappen?
similarity_rankings = build_similarity_ranking(dna_vectors)

TARGET = 'VER'
similar_to_ver = similarity_rankings[TARGET]

print(f'Drivers most similar to {race_data[TARGET].full_name}:\n')
print(f'{"Rank":<5} {"Code":<6} {"Driver":<25} {"Similarity":<12} {"Shared Strengths"}')
print('─' * 75)
for rank, s in enumerate(similar_to_ver, 1):
    shared = ', '.join(s.shared_strengths[:3]) or 'none'
    print(f'{rank:<5} {s.driver_b:<6} {race_data[s.driver_b].full_name:<25} {s.similarity:<12.4f} {shared}')

In [ ]:
# Full DNA profiles
profiles = build_dna_profiles(dna_vectors, clustering, similarity_rankings)

# Print top 3 finishers
for code, _ in sorted_drivers[:3]:
    if code in profiles:
        print(profiles[code].summary())

---
## Summary

| Module | What it produced |
|---|---|
| Data Pipeline | Parsed 20 drivers, 57 laps each, 3 stints per driver |
| Rating Engine | 5-pillar composite scores normalised across the field |
| Strategy Simulator | Pit timing verdicts, undercut sims, SC luck index |
| Driver DNA | 8-dimensional identity vectors, KMeans clusters, similarity map |

The full interactive dashboard runs via:
```bash
streamlit run f1pitwall/app/main.py
```